# Train a Tiny Model

Companion notebook to *Build AI, Understand AI*, Part VI, Chapter 1.

In one notebook, on a CPU or a free Colab GPU, you will train a tiny
character-level language model from nothing: tokenise text into numbers,
watch a loss fall, sample from the model before and after training, and
then overtrain it on purpose to watch memorisation happen.

**Simplified, not production.** This model will babble. That is the point:
every mechanism in a frontier model's training run (batches, loss,
gradients, sampling temperature) is present here at a size you can read.

Needs only PyTorch: `pip install torch` (preinstalled on Colab).
Expected time: the training cell runs in under a minute on a laptop CPU
for the defaults below; raise STEPS when you are ready to wait longer.


## Step 0: Settings

Everything you might change lives here. `SEED` makes runs comparable:
change one setting at a time and you can attribute what changed.


In [1]:
import time

import torch
import torch.nn as nn

SEED = 42
BLOCK = 32        # characters of context the model sees at once
BATCH = 32        # examples per training step
D_MODEL = 64      # width of the model's internal vectors
STEPS = 600       # training steps for the first run
LEARNING_RATE = 3e-3

torch.manual_seed(SEED)
print("torch", torch.__version__)


torch 2.13.0


## Step 1: Text becomes numbers

A language model never sees letters. It sees integers from a vocabulary.
Character level is the smallest honest version: one integer per distinct
character in our training text, which is built from sentences about this
very book so you can audit every word it could possibly learn.


In [2]:
sentences = [
    "the model proposes and the code disposes",
    "a judge marks work against criteria it did not write",
    "the transcript is resent on every call",
    "a tool is a function the model may ask for",
    "the index is a snapshot and snapshots go stale",
    "typed answers beat paragraphs when code is listening",
    "the registry is the complete list of what asking can achieve",
    "confidence is a claim to calibrate not a fact to obey",
    "a failed stage leaves its error in the out tray",
    "the loop keeps the best so far and logs the rest",
]
# Hold out whole sentences for validation. Splitting repeated text at a
# character offset would leak the same sentences into both sides and
# flatter the validation loss (try it later and watch the gap vanish).
train_text = ". ".join(sentences[:8] * 12) + "."
val_text = ". ".join(sentences[8:] * 12) + "."
text = train_text + " " + val_text  # vocabulary sees every character used
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)
print(f"train: {len(train_text)} chars from 8 sentences; val: {len(val_text)} chars from 2 held-out sentences")
print(f"vocabulary: {len(chars)} symbols")


train: 4787 chars from 8 sentences; val: 1187 chars from 2 held-out sentences
vocabulary: 26 symbols


Two sentences are held out entirely as the sealed validation set:
the trainer never trains on them. It is how we will tell learning
from memorising, the same sealed-eval discipline as the adapter
notebook that follows. (Leakage check worth doing later: split one
repeated text at a character offset instead, and watch validation
loss lie to you.)


In [3]:
train_data, val_data = encode(train_text), encode(val_text)

def get_batch(source):
    ix = torch.randint(len(source) - BLOCK, (BATCH,))
    x = torch.stack([source[i : i + BLOCK] for i in ix])
    y = torch.stack([source[i + 1 : i + BLOCK + 1] for i in ix])
    return x, y

x, y = get_batch(train_data)
print("one input row decoded :", "".join(itos[i] for i in x[0].tolist()))
print("its target row decoded:", "".join(itos[i] for i in y[0].tolist()))
print("the target is the input shifted by one character: next-token prediction")


one input row decoded : code disposes. a judge marks wor
its target row decoded: ode disposes. a judge marks work
the target is the input shifted by one character: next-token prediction


## Step 2: A model small enough to read

One transformer encoder layer: turn token ids (plus positions) into
vectors, let them attend to earlier positions only (the causal mask),
and score every vocabulary symbol as the possible next character.


In [4]:
class TinyModel(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab, D_MODEL)
        self.pos = nn.Embedding(BLOCK, D_MODEL)
        layer = nn.TransformerEncoderLayer(D_MODEL, 4, 128, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, 1)
        self.head = nn.Linear(D_MODEL, vocab)

    def forward(self, x):
        t = x.shape[1]
        mask = torch.triu(torch.full((t, t), float("-inf")), 1)
        h = self.emb(x) + self.pos(torch.arange(t))
        return self.head(self.tr(h, mask=mask))

model = TinyModel(len(chars))
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")


parameters: 38,874


## Step 3: Sample before training

An untrained model's weights are near-random, so its samples are
character noise. Keep this output: it is your baseline, the control
group every later claim is measured against.


In [5]:
@torch.no_grad()
def sample(model, start="the ", n=90, temperature=0.8):
    model.eval()
    ctx = torch.tensor([[stoi[c] for c in start]])
    for _ in range(n):
        logits = model(ctx[:, -BLOCK:])
        probs = torch.softmax(logits[0, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1).view(1, 1)
        ctx = torch.cat([ctx, nxt], dim=1)
    model.train()
    return "".join(itos[i] for i in ctx[0].tolist())

print(repr(sample(model)))


'the ifolhpydttyctri.jfemckylhcr.dpkxu.fsdscouotlf x deorxxxaoxdyilatyleumndgniblxwudunswismuxc'


## Step 4: Train

Each step: guess the next character everywhere at once, measure the
average surprise (cross-entropy loss), let the optimiser nudge every
weight a little downhill. Watch both losses. On this tiny, repetitive
corpus the training loss falls fast while the held-out loss rises:
the model is memorising eight sentences, not learning English, and
the sealed sentences catch it in the act. That divergence, visible
within a hundred steps, is the single most useful training graph
in this book.


In [6]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

@torch.no_grad()
def val_loss():
    model.eval()
    x, y = get_batch(val_data)
    out = loss_fn(model(x).view(BATCH * BLOCK, -1), y.view(BATCH * BLOCK))
    model.train()
    return out.item()

history = []
t0 = time.time()
for step in range(STEPS + 1):
    if step % 100 == 0:
        x, y = get_batch(train_data)
        tl = loss_fn(model(x).view(BATCH * BLOCK, -1), y.view(BATCH * BLOCK)).item()
        vl = val_loss()
        history.append((step, round(tl, 3), round(vl, 3)))
        print(f"step {step:4}  train {tl:.3f}  val {vl:.3f}")
    x, y = get_batch(train_data)
    loss = loss_fn(model(x).view(BATCH * BLOCK, -1), y.view(BATCH * BLOCK))
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"trained in {time.time() - t0:.1f}s")


step    0  train 3.322  val 3.350


step  100  train 1.500  val 3.027


step  200  train 0.606  val 4.477


step  300  train 0.413  val 5.260


step  400  train 0.332  val 5.553


step  500  train 0.248  val 5.862


step  600  train 0.233  val 6.154
trained in 3.5s


## Step 5: Sample after training, at two temperatures

Temperature divides the scores before sampling: low values make the
model conservative (nearer its training text), high values make it
adventurous (nearer the noise you saw in Step 3). Same weights, same
knowledge, different risk appetite at the door.


In [7]:
for temp in (0.5, 1.2):
    print(f"--- temperature {temp}")
    print(repr(sample(model, temperature=temp)))


--- temperature 0.5
'the model proposes and the code disposes. a judge marks work against criteria it did not write'
--- temperature 1.2
'the model proposes oses and the code disposes. a judge marks work against criteria it did not '


## Step 6: Overtrain on purpose

Run five times more steps. Training loss dives toward memorising the
corpus; check whether validation loss follows or refuses. Then sample:
verbatim regurgitation is memorisation wearing generation's clothes,
and it is why serious training reports held-out numbers, and why the
adapter notebook keeps its eval set sealed.


In [8]:
for step in range(STEPS * 4):
    x, y = get_batch(train_data)
    loss = loss_fn(model(x).view(BATCH * BLOCK, -1), y.view(BATCH * BLOCK))
    opt.zero_grad()
    loss.backward()
    opt.step()
print("final train loss:", round(loss.item(), 3), " val loss:", round(val_loss(), 3))
print(repr(sample(model, start="the model ", temperature=0.5)))


final train loss: 0.153  val loss: 7.984
'the model may ask for. the index is a snapshot and snapshots go stale. typed answers beat paragraphs'


## What to try next

- Paste in your own text file (a poem, your commit messages) at Step 1.
  The vocabulary grows; watch what that does to the first loss value.
- Halve `D_MODEL`, then double it. Which changed the samples more, width
  or steps?
- Start sampling from a word the corpus never starts a sentence with.
- Karpathy's line, for the full-size version of this afternoon:
  *Zero to Hero* for the concepts, nanochat for the modern codebase.
